In [ ]:
#
# For licensing see accompanying LICENSE file.
# Copyright (C) 2025 Apple Inc. All Rights Reserved.
#

In [ ]:
import os
import hydra
import torch
from einops import rearrange, repeat
import omegaconf
from omegaconf import DictConfig, OmegaConf
from utils.modelrunner import ModelRunner
from torchvision.utils import save_image

In [ ]:
# define the sampler
from model.sampler import ODESampler

sampler = ODESampler(
    num_timesteps=10,
    cfg_scale=1.0,
    t_eps=1e-4,
    sample_step="euler",
    eval_ema=True,
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = OmegaConf.load("configs/model/architecture/pointdit_demo.yaml")
model = hydra.utils.instantiate(cfg)
print(model)

from torch.optim.swa_utils import AveragedModel
model_ema = AveragedModel(
    model,
    multi_avg_fn=torch.optim.swa_utils.get_ema_multi_avg_fn(
        0.999
    ),
    use_buffers=True,
)

# Load the model checkpoint
# model.load_state_dict(
#     torch.load(CKPT_PATH, map_location="cpu")["state_dict"],
#     strict=False,
# )
# model_ema.load_state_dict(
#     torch.load(EMA_CKPT_PATH, map_location="cpu")["state_dict"],
#     strict=False,
# )

model = model.to(device)
model_ema = model_ema.to(device)
model.eval()
model_ema.eval()

In [ ]:
from model.path import LinearPath

path = LinearPath()

sampler.setup(
    model=model,
    model_ema=model_ema,
    path=path,
)

In [ ]:

num_samples = 1
num_points = 16384
num_points_train = 2048

# we keep the context the same point as the low resolution in training
# so that we can use whatever resolution we want to sample as queries

y_noise = torch.randn(
    num_samples, num_points, 3, device=device
)
x = y_noise.clone()

context_mask = torch.zeros((num_points, ), device=device, dtype=torch.bool)
context_mask[:num_points_train] = True  # Use the first 2048 points as context as in training

y_sampled = sampler.sample(
    query_x=x,
    query_y_sampled=y_noise,
    context_mask=context_mask,
)
print(f"y_sampled: {y_sampled.shape}")


from evaluator.pointcloud_evaluator import write_pointcloud
# Save pointclouds to file
for i, sample in enumerate(y_sampled):
    write_pointcloud(f"pcd_{i}.ply", sample.cpu().numpy())
